# 01 – IPL Data Inspection (Phase 2)

**Goal:** open `matches.csv` and `deliveries.csv` and *look* at them before cleaning anything. Don't change any data yet. A good analyst first understands the data.

**How to run:** click a cell and press **Shift + Enter**. Run the cells from top to bottom.

## Step 1 – Import libraries
- **pandas** (`pd`) works with tables. A table in pandas is called a **DataFrame**.
- **numpy** (`np`) does fast maths on numbers.
- `as pd` / `as np` are short nicknames, a convention every analyst follows.

In [ ]:
import pandas as pd
import numpy as np
import os

# Show all columns when printing (otherwise pandas hides the middle ones with '...')
pd.set_option("display.max_columns", None)

print("pandas version:", pd.__version__)
print("numpy version :", np.__version__)

**Expected output:** two version numbers, e.g. `pandas version: 2.2.x`. If you see `ModuleNotFoundError`, run `pip install -r requirements.txt` in the terminal and restart the kernel.

## Step 2 – Set the file paths
The notebook is in `notebooks/`, and the data is one level up in `data/raw/`. `..` means "go up one folder".

**Google Colab users:** change `DATA_DIR` to `"/content/"` after uploading the CSVs.

In [ ]:
DATA_DIR = "../data/raw/"          # Colab: "/content/"
MATCHES_PATH = os.path.join(DATA_DIR, "matches.csv")
DELIVERIES_PATH = os.path.join(DATA_DIR, "deliveries.csv")

for path in [MATCHES_PATH, DELIVERIES_PATH]:
    if os.path.exists(path):
        print("Found   :", path)
    else:
        print("MISSING :", path, "-> download it first (see data/raw/README.md)")

## Step 3 – Read the CSV files
`pd.read_csv()` loads a CSV into a DataFrame. `low_memory=False` tells pandas to read the whole file before deciding each column's data type, which avoids a `DtypeWarning`.

In [ ]:
matches = pd.read_csv(MATCHES_PATH)
deliveries = pd.read_csv(DELIVERIES_PATH, low_memory=False)
print("Loaded successfully!")

## Step 4 – Size of each dataset
`.shape` returns `(rows, columns)`.

**Expected:** matches ≈ `(1095, 20)`, deliveries ≈ `(260920, 17)`. The exact numbers depend on which dataset version you downloaded.

In [ ]:
print("matches    shape:", matches.shape)
print("deliveries shape:", deliveries.shape)

## Step 5 – Look at the first rows
`.head()` shows the first 5 rows. Compare what you see with the column table in `docs/phase2_dataset_guide.md`.

In [ ]:
matches.head()

In [ ]:
deliveries.head()

## Step 6 – Column names

In [ ]:
print("matches columns:\n", list(matches.columns), "\n")
print("deliveries columns:\n", list(deliveries.columns))

## Step 7 – Data types and non-null counts
`.info()` lists every column with:
- **Non-Null Count**: how many rows have a value. If this is lower than the total rows, the column has missing values.
- **Dtype**: `int64` = whole numbers, `float64` = decimals, `object` = text.

Note that `date` is shown as `object` (text). We convert it to a real date in Phase 3.

In [ ]:
matches.info()

In [ ]:
deliveries.info()

## Step 8 – Missing values per column
`.isnull()` marks each empty cell as True, and `.sum()` counts the True values in each column.

In [ ]:
missing = matches.isnull().sum()
print("matches – columns with missing values:")
print(missing[missing > 0].sort_values(ascending=False))

In [ ]:
missing_d = deliveries.isnull().sum()
print("deliveries – columns with missing values:")
print(missing_d[missing_d > 0].sort_values(ascending=False))

**Why do `player_dismissed`, `extras_type` and `fielder` have so many blanks?** Most balls are *not* wickets or extras, so these cells are empty. That is correct, not an error. In `matches`, `method` is empty for most rows because few matches are affected by rain. Understanding *why* a value is missing is a common viva question.

## Step 9 – Duplicate rows

In [ ]:
print("Duplicate rows in matches   :", matches.duplicated().sum())
print("Duplicate rows in deliveries:", deliveries.duplicated().sum())
print("Is matches.id unique?       :", matches["id"].is_unique)

## Step 10 – Quick statistics for number columns
`.describe()` gives count, mean, min, max and quartiles. Look for impossible values, e.g. `batsman_runs` above 6.

In [ ]:
deliveries[['batsman_runs', 'extra_runs', 'total_runs', 'is_wicket']].describe()

## Step 11 – Look at category values
This is how we find the **team name problem** we fix in Phase 3.

In [ ]:
print("Seasons:\n", matches["season"].unique(), "\n")
print("Toss decisions:\n", matches["toss_decision"].value_counts(), "\n")
print("Result types:\n", matches["result"].value_counts())

In [ ]:
all_teams = sorted(set(matches["team1"]).union(set(matches["team2"])))
print(len(all_teams), "team names found:")
for t in all_teams:
    print(" -", t)

**What to notice:** some franchises appear under more than one name (e.g. *Delhi Daredevils* / *Delhi Capitals*, *Kings XI Punjab* / *Punjab Kings*, *Rising Pune Supergiant* / *Rising Pune Supergiants*, *Royal Challengers Bangalore* / *Royal Challengers Bengaluru*). We will standardise these in Phase 3.

## Step 12 – Check the relationship between the two files
Every `deliveries.match_id` should exist in `matches.id` (foreign key → primary key).

In [ ]:
match_ids = set(matches["id"])
delivery_match_ids = set(deliveries["match_id"])

orphans = delivery_match_ids - match_ids
print("Matches in matches.csv              :", len(match_ids))
print("Distinct match_id in deliveries.csv :", len(delivery_match_ids))
print("match_ids with no parent match      :", len(orphans))
print("Average balls per match             :", round(len(deliveries) / len(delivery_match_ids), 1))

**Expected:** `match_ids with no parent match : 0`, and roughly 238 balls per match. This shows that the files can be joined safely on `matches.id = deliveries.match_id`.

A few matches might have no deliveries at all (abandoned matches). That is acceptable.

## Step 13 – A first taste of analysis
A preview of Phase 3. This shows the 5 highest run scorers of all time.

In [ ]:
top_scorers = (deliveries.groupby("batter")["batsman_runs"]
               .sum()
               .sort_values(ascending=False)
               .head(5))
top_scorers

## Summary – what you learned
1. matches.csv has one row per match and deliveries.csv has one row per ball.
2. They are linked with `matches.id` → `deliveries.match_id` (1:N).
3. Issues to fix in Phase 3: missing values (`city`, `winner`, ...), duplicate team names, `date` stored as text, inconsistent `season` values.

✅ **Phase 2 is complete.** Commit your work to GitHub, then move on to Phase 3.